In [ ]:
import pandas as pd
from pathlib import Path

parent_dir = Path.cwd().parents[1]
file_path = parent_dir / "Dataset" / "research_corpus"

files = [
    file_path / "research_1.txt",
    file_path / "research_2.txt",
    file_path / "research_3.txt",
    file_path / "research_4.txt",
    file_path / "research_5.txt"
]

data = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    data.append({
        "file_name": file.name,
        "text":text
    })

df = pd.DataFrame(data)

print(df.shape)
print(df.head())

(5, 2)
        file_name                                               text
0  research_1.txt  The purposes of this article are to position m...
1  research_2.txt  The purposes of this article are to position m...
2  research_3.txt  Now I see it, now I don’t:\nresearcher’s posit...
3  research_4.txt  DEVELOPMENTAL RESEARCH: STUDIES OF\nINSTRUCTIO...
4  research_5.txt  Abstract\nCronbach’s alpha is a statistic comm...


In [8]:
pattern_one = r'[^\w\s]'
pattern_two = r'\s+'

df["clean_text"] = (
    df["text"]
    .str.replace(pattern_one, ' ', regex=True)
    .str.replace(pattern_two, ' ', regex=True)
    .str.strip()
)

In [9]:
df["clean_text"] = df["clean_text"].apply(str.lower)

In [10]:
from nltk import pos_tag, word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import nltk

for resource, location in [
    ("punkt_tab", "tokenizers/punkt_tab"),
    ("averaged_perceptron_tagger_eng", "taggers/averaged_perceptron_tagger_eng"),
    ("wordnet", "corpora/wordnet.zip"),
]:
    try:
        nltk.data.find(location)
    except LookupError:
        nltk.download(resource, raise_on_error=True)

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = word_tokenize(text)
    tagged_words = pos_tag(words)

    lemmas = []

    for word, tag in tagged_words:
        if tag.startswith('J'):
            pos = wordnet.ADJ
        elif tag.startswith('V'):
            pos = wordnet.VERB
        elif tag.startswith('N'):
            pos = wordnet.NOUN
        elif tag.startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN

        lemmas.append(lemmatizer.lemmatize(word, pos))

    return ' '.join(lemmas)

In [11]:
df['text_lemmatized'] = df['clean_text'].apply(lemmatize_text)

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df["stop_removed"] = df["text_lemmatized"].apply(
    lambda x: ' '.join([w for w in x.split() if w.lower() not in ENGLISH_STOP_WORDS])
)

## 3.3 Summarising the first article

An **extractive TF-IDF summarisation** technique is used for `research_1.txt`. The article is split into sentences, then TF-IDF assigns a weight to each word according to its importance in the article. A sentence's score is the average TF-IDF weight of its words, so the highest-scoring sentences form the summary. The chart visualises the ten most frequent meaningful terms in the first article.

In [ ]:
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

first_article = df.loc[df['file_name'].eq('research_1.txt')].iloc[0]
sentences = [sentence.strip() for sentence in sent_tokenize(first_article['text']) if sentence.strip()]

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sentences)
sentence_scores = np.asarray(tfidf_matrix.mean(axis=1)).ravel()

summary_length = min(5, len(sentences))
top_sentence_indices = np.argsort(sentence_scores)[-summary_length:]
summary = [sentences[index] for index in sorted(top_sentence_indices)]

print('Extractive TF-IDF summary of research_1.txt:\n')
for number, sentence in enumerate(summary, start=1):
    print(f'{number}. {sentence}\n')

In [ ]:
import matplotlib.pyplot as plt

term_frequencies = np.asarray(tfidf_matrix.sum(axis=0)).ravel()
terms = vectorizer.get_feature_names_out()
top_term_indices = term_frequencies.argsort()[-10:][::-1]

plt.figure(figsize=(10, 5))
plt.barh(terms[top_term_indices][::-1], term_frequencies[top_term_indices][::-1], color='steelblue')
plt.title('Ten most important terms in research_1.txt (TF-IDF)')
plt.xlabel('Total TF-IDF score')
plt.ylabel('Term')
plt.tight_layout()
plt.show()

## 3.4 Aggregate summary of the remaining articles

The four remaining articles (`research_2.txt` to `research_5.txt`) are summarised together using **Latent Semantic Analysis (LSA)**. Unlike the TF-IDF ranking in Question 3.3, LSA uses singular-value decomposition to identify the main latent topics across all sentences in the combined corpus. The sentence most strongly associated with each topic is selected to create a concise, non-redundant aggregate summary. The heatmap visualises the five highest-weighted terms for each latent topic.

In [ ]:
from sklearn.decomposition import TruncatedSVD

remaining_articles = df.loc[~df['file_name'].eq('research_1.txt')].copy()
aggregate_sentences = []

for _, article in remaining_articles.iterrows():
    aggregate_sentences.extend(
        sentence.strip() for sentence in sent_tokenize(article['text']) if sentence.strip()
    )

lsa_vectorizer = TfidfVectorizer(stop_words='english', min_df=2)
aggregate_tfidf = lsa_vectorizer.fit_transform(aggregate_sentences)

n_topics = min(4, aggregate_tfidf.shape[0] - 1, aggregate_tfidf.shape[1] - 1)
lsa_model = TruncatedSVD(n_components=n_topics, random_state=42)
sentence_topic_strengths = lsa_model.fit_transform(aggregate_tfidf)

selected_indices = []
for topic_index in range(n_topics):
    ranked_indices = np.argsort(sentence_topic_strengths[:, topic_index])[::-1]
    selected_index = next(index for index in ranked_indices if index not in selected_indices)
    selected_indices.append(selected_index)

aggregate_summary = [aggregate_sentences[index] for index in selected_indices]

print('LSA aggregate summary of research_2.txt to research_5.txt:\n')
for number, sentence in enumerate(aggregate_summary, start=1):
    print(f'{number}. {sentence}\n')

In [ ]:
terms = lsa_vectorizer.get_feature_names_out()
top_terms_per_topic = 5
topic_labels = [f'Topic {index + 1}' for index in range(n_topics)]

topic_terms = []
for component in lsa_model.components_:
    top_indices = np.argsort(np.abs(component))[-top_terms_per_topic:][::-1]
    topic_terms.append([(terms[index], component[index]) for index in top_indices])

all_top_terms = list(dict.fromkeys(term for topic in topic_terms for term, _ in topic))
term_weights = np.zeros((n_topics, len(all_top_terms)))
for topic_index, topic in enumerate(topic_terms):
    for term, weight in topic:
        term_weights[topic_index, all_top_terms.index(term)] = weight

plt.figure(figsize=(max(10, len(all_top_terms) * 0.6), 5))
image = plt.imshow(term_weights, cmap='Blues', aspect='auto')
plt.colorbar(image, label='LSA term weight')
plt.xticks(range(len(all_top_terms)), all_top_terms, rotation=45, ha='right')
plt.yticks(range(n_topics), topic_labels)
plt.title('Dominant terms in topics across research_2.txt to research_5.txt')
plt.xlabel('Term')
plt.ylabel('Latent topic')
plt.tight_layout()
plt.show()